# The jupyter notebook to check whether the model generated is structurally valid and correct

We now have generated the new model using `export_savedmodel.py`; but we of course cannot be sure just by that code to see whether the model made in ONNX is the same as the model we originally had in Keras. We begin by seeing whether the model is structurally valid.

In [ ]:
import onnx
model = onnx.load("das_model.onnx")
onnx.checker.check_model(model)
print("ONNX model is valid")

ONNX model is valid


We now have verified that at least the model is structurally valid, now we go on to see whether the models descrribe the same. The most important rule of verifying this is we have to verify that the models have the same input tensors, same shape and are in inference mode.

In [ ]:
import numpy as np
import tensorflow as tf
import onnxruntime as ort
from export_savedmodel import custom_objects #we import the same custom objects from export_savedmodel as we needed for converting the mdoel

np.random.seed(0)
x = np.random.randn(1,1024,1).astype(np.float32)

#Then we run the inference with the ONNX model, first we have to make sure we are in inference mode:
tf.keras.backend.clear_session()

m = tf.keras.models.load_model(
    "20251116_115659_model.h5",
    custom_objects=custom_objects,
    compile=False
)

y_tf = m(x, training=False)

if isinstance(y_tf, (list, tuple)):
    y_tf = [t.numpy() for t in y_tf]
else:
    y_tf = [y_tf.numpy()]

sess = ort.InferenceSession(
    "das_model.onnx",
    providers=["CPUExecutionProvider"]
)

print("ONNX inputs:", [i.name for i in sess.get_inputs()])
print("ONNX outputs:", [o.name for o in sess.get_outputs()])

y_onnx = sess.run(None, {"input_1": x})

2026-01-26 13:00:26.721363: I tensorflow/core/grappler/devices.cc:66] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
2026-01-26 13:00:26.721614: I tensorflow/core/grappler/clusters/single_machine.cc:357] Starting new session
2026-01-26 13:00:27,363 - INFO - Using tensorflow=2.13.0, onnx=1.17.0, tf2onnx=1.16.1/15c810
2026-01-26 13:00:27,364 - INFO - Using opset <onnx, 13>
2026-01-26 13:00:27,729 - INFO - Computed 0 values for constant folding
2026-01-26 13:00:28,391 - INFO - Optimizing ONNX model
2026-01-26 13:00:32,352 - INFO - After optimization: Cast -35 (37->2), Concat -2 (4->2), Const -145 (230->85), Gather -1 (2->1), GlobalMaxPool +15 (0->15), Identity -2 (2->0), ReduceMax -15 (16->1), ReduceProd -2 (2->0), Reshape -30 (35->5), Shape -1 (2->1), Slice -1 (2->1), Squeeze -1 (32->31), Transpose -33 (67->34), Unsqueeze -5 (37->32)
2026-01-26 13:00:32,409 - INFO - 
2026-01-26 13:00:32,409 - INFO - Successfully converted TensorFlow model das_savedmodel to ONNX
2

ONNX inputs: ['input_1']
ONNX outputs: ['up_sampling1d']


Now we check whether the outputs are numerically within the tolerances:

In [ ]:
for i, (a, b) in enumerate(zip(y_tf, y_onnx)):
    print(f"\nOutput {i}")
    print("shape keras:", a.shape)
    print("shape onnx:", b.shape)

    max_abs = np.max(np.abs(a - b))
    mean_abs = np.mean(np.abs(a - b))

    print("max (diff):", max_abs)
    print("mean (diff):", mean_abs)

    np.testing.assert_allclose(
        a, b,
        rtol=1e-4,
        atol=1e-5
    )



Output 0
shape keras: (1, 1024, 2)
shape onnx: (1, 1024, 2)
max (diff): 2.3841858e-07
mean (diff): 5.7858415e-08


The `max(diff)` value here is exactly one difference float32 ULP $2^{-22}$, which means the values are as good as perfectly the same. Just to make sure that they're close to the same layer-by-layer

In [ ]:
layer_names = [l.name for l in m.layers]

debug_model = tf.keras.Model(
    inputs=m.input,
    outputs=[l.output for l in m.layers]
)

keras_acts = debug_model(x, training=False)

ort_outputs = sess.run(
    [o.name for o in sess.get_outputs()],
    {"input_1": x}
)

In [ ]:
np.testing.assert_allclose(y_tf, y_onnx, rtol=1e-4, atol=1e-5)

This has no output, therefore we can conclude that the layers are the same, therefore we know the models are the same.